# SpamGuard — Makine Öğrenmesi Karşılaştırması

Yaser Durdu için çalışma dosyası. İngilizce SMS üzerinde spam/normal sınıflandırması. Veri: Almeida & Hidalgo (2011), UCI SMS Spam Collection, https://doi.org/10.24432/C5CC84, CC BY 4.0.

Bu defter gerçek veriyle çalıştırılmıştır. Colab’a yükleyip Çalışma zamanı > Tümünü çalıştır ile yeniden üretilebilir. Bu ML defterinde etkileşimli input hücresi yoktur; mini uygulama ayrı Python çalışmasındadır.

## Problem, veri ve hazırlık
Amaç, İngilizce SMS içeriğinden spam etiketini tahmin etmektir. Etiketi kaynak veri sağlar. Etiket = spam/ham; sayısal hedef = 1/0. Kimlik veya dosya adı özellik değildir. Tekrarları bölmeden önce çıkarırız. Eğitim %60, doğrulama %20, test %20. Test bölümünü model ayarı için kullanmayız.

In [ ]:
from pathlib import Path
import io, zipfile, urllib.request, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.dummy import DummyClassifier

# Colab: ham veri yoksa resmi UCI adresinden indir.
raw_path = Path('SMSSpamCollection')
if not raw_path.exists():
    local = Path('../data/SMSSpamCollection')
    if local.exists(): raw_path = local
    else:
        url='https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'
        with urllib.request.urlopen(url, timeout=60) as response:
            archive=zipfile.ZipFile(io.BytesIO(response.read()))
        raw_path.write_bytes(archive.read('SMSSpamCollection'))
raw=pd.read_csv(raw_path,sep='\t',names=['label','text'],quoting=3,keep_default_na=False)
raw['key']=raw.text.map(lambda s: re.sub(r'\s+',' ',s.strip().lower()))
conflicts=raw.groupby('key').label.nunique()
conflicts=set(conflicts[conflicts>1].index)
df=raw[(raw.key!='') & raw.label.isin(['ham','spam']) & ~raw.key.isin(conflicts)].drop_duplicates('key').copy().reset_index(drop=True)
df['target']=(df.label=='spam').astype(int)
def features(text):
    letters=[c for c in text if c.isalpha()]
    return [len(text),len(text.split()),sum(c.isdigit() for c in text),text.count('!'),sum(c.isupper() for c in letters)/max(len(letters),1),len(re.findall(r'https?://|www\.',text.lower()))]
FEATURES=['character_count','word_count','digit_count','exclamation_count','uppercase_ratio','url_count']
df[FEATURES]=pd.DataFrame(df.text.map(features).tolist(),index=df.index)
train,rest=train_test_split(df.index,test_size=.4,random_state=42,stratify=df.target)
val,test=train_test_split(rest,test_size=.5,random_state=42,stratify=df.loc[rest,'target'])
df['split']='train'
df.loc[val,'split']='validation'
df.loc[test,'split']='test'
df.drop(columns='key').to_csv('sms_features.csv',index=False)
print('Ham/temiz:',len(raw),len(df))
print('Sınıf sayıları:',df.label.value_counts().to_dict())
print('Bölüm sayıları:',df.split.value_counts().to_dict())
print('Eksik değer:',df.isna().sum().sum())
print(df[['text','label']].head().to_string(index=False))


## Ön İşleme Kararları ve Gerekçeleri
- Eksik/boş/geçersiz kayıt ve çelişen etiketler kontrol edilir; bulunmadığı için yapay doldurma uygulanmaz. 415 normalize tam tekrar çıkarılır.
- Etiket kodlaması ham=0, spam=1. Metin TF-IDF ile sayısallaşır; kayıt numarası ve etiket özellik değildir.
- Uzun mesajlar hatalı sayılıp otomatik silinmez; gerçek spam örnekleri olabilir. Histogramın 1.000 karakterlik gösterim sınırı veri filtresi değildir.
- TF-IDF'de küçük harf, en az iki ASCII harf/rakam/alt çizgiden oluşan token, sublinear_tf ve L2 normalizasyon kullanılır. Bu token kuralı £ gibi işaretleri ve tek karakterleri dışarıda bırakır; bilgi kaybı yaratabilir.
- TF-IDF için ek StandardScaler kullanılmaz; vektörler zaten L2 normalize edilir. Sayısal karar ağacında ölçekleme gerekmez.
- Görüntü yeniden boyutlandırma ve veri artırma bu metin projesi için gerekli değildir; çoğaltma veya yapay örnek ekleme yapılmaz. Dengesizlik LR'de balanced ağırlık, stratified ayrım ve spam F1 ile ele alınır.
- %60/%20/%20 stratified ayrım, seed=42. Sözlük ve IDF yalnızca eğitimde fit edilir. Hiperparametre seçimi doğrulama F1, nihai ölçüm sabit test üzerinde yapılır.

## Keşif
Eksik değer kontrolü, describe, groupby ve üç etiketli grafik. Normal mesajlar çoğunlukta olduğundan yalnızca doğrulukla karar vermeyiz. Özellik birimleri veri sözlüğünde açıklanır.

In [ ]:
print(df[FEATURES].describe().round(3).to_string())
print(df.groupby('label')[FEATURES].mean().round(3).to_string())
fig,axes=plt.subplots(1,3,figsize=(15,4))
df.label.value_counts().reindex(['ham','spam']).plot.bar(ax=axes[0],color=['#168a78','#e08245'],rot=0)
axes[0].set(title='Sınıf dağılımı',xlabel='Sınıf',ylabel='Mesaj sayısı')
for label in ['ham','spam']:
    axes[1].hist(df.loc[df.label==label,'character_count'],bins=np.arange(0,1000,25),alpha=.5,label=label,density=True)
axes[1].legend()
axes[1].set(title='Mesaj uzunluğu',xlabel='Karakter',ylabel='Yoğunluk')
df.groupby('label')[['digit_count','exclamation_count','url_count']].mean().T.plot.bar(ax=axes[2],rot=0,color=['#168a78','#e08245'])
axes[2].set(title='Ortalama özellikler',xlabel='Özellik',ylabel='Ortalama')
plt.tight_layout()
plt.savefig('veri_analizi.png',dpi=140)


### Bulgular
1. Normal 4.517, spam 642 mesaj: çoğunluk temel modelinin doğruluğu %87,60, spam F1 sıfırdır.
2. Spamın ortalama uzunluğu 137,45, normalin 70,90 karakterdir; dağılımlar örtüşür.
3. Spamda ortalama rakam sayısı 15,44, normalde 0,30'dur; faydalı aday özellik olabilir.
4. Spamda URL örüntüsü ortalaması 0,156, normalde 0,0002; ünlem ortalaması sırasıyla 0,69 ve 0,18'dir.
5. Bu ilişkiler tek başına spam kanıtı değildir. EDA tüm veri üzerinde betimleyicidir; hiperparametre kararı test metrikleriyle verilmez.

## Model tasarımı
TF-IDF en fazla 5.000 tek kelime özelliği, sublinear_tf=True ve L2 normalizasyon kullanır. Sözlük yalnızca eğitim verisinden öğrenilir. Logistic Regression C ve Naive Bayes alpha doğrulama F1 ile seçilir. LR seçimi testten önce doğrulama sonucuna dayanır. İki model 0,5 eşikle karşılaştırılır.

**Neden bu modeller?** Logistic Regression yüksek boyutlu seyrek TF-IDF girdisinde hesaplama açısından hafiftir ve katsayıları terim katkılarını açıklamayı sağlar. Multinomial Naive Bayes negatif olmayan metin özellikleriyle hızlı bir karşılaştırmadır; kelimelerin koşullu bağımsızlığı varsayımı sınırlılığıdır. Çoğunluk modeli, dengesiz veride yüksek doğruluğun tek başına yeterli olmadığını gösterir. DL mimarisi kullanmak zorunlu değildir; küçük veri ve açıklanabilirlik hedefi için klasik ML tercih edildi. C düzenlileştirme gücünün tersini, alpha ise Naive Bayes yumuşatmasını kontrol eder. Seçilen LR'nin doğrulama F1 üstünlüğü çok küçüktür (%92,1875 / %92,1811); kesin veya istatistiksel olarak anlamlı üstünlük iddia etmeyiz.

In [ ]:
y=df.target
vec=TfidfVectorizer(lowercase=True,token_pattern=r'[a-z0-9_]{2,}',sublinear_tf=True,max_features=5000)
Ztrain=vec.fit_transform(df.loc[train,'text'])
Zval=vec.transform(df.loc[val,'text'])
Ztest=vec.transform(df.loc[test,'text'])
lr_candidates=[]
for C in [.5,1.,2.,4.]:
    model=LogisticRegression(C=C,class_weight='balanced',max_iter=1000,random_state=42).fit(Ztrain,y.loc[train])
    lr_candidates.append((f1_score(y.loc[val],model.predict(Zval)),C,model))
_,C,lr=max(lr_candidates,key=lambda x:x[0])
nb_candidates=[]
for alpha in [.1,.5,1.]:
    model=MultinomialNB(alpha=alpha).fit(Ztrain,y.loc[train])
    nb_candidates.append((f1_score(y.loc[val],model.predict(Zval)),alpha,model))
_,alpha,nb=max(nb_candidates,key=lambda x:x[0])
baseline=DummyClassifier(strategy='most_frequent').fit(Ztrain,y.loc[train])
models={'Baseline':baseline,'Logistic Regression':lr,'Naive Bayes':nb}
rows=[]
for name,model in models.items():
    pred=model.predict(Ztest)
    rows.append({'model':name,'accuracy':accuracy_score(y.loc[test],pred),'precision':precision_score(y.loc[test],pred,zero_division=0),'recall':recall_score(y.loc[test],pred),'f1':f1_score(y.loc[test],pred)})
results=pd.DataFrame(rows)
print('Seçilen C ve alpha:',C,alpha)
print(results.to_string(index=False))
results.to_csv('model_karsilastirma.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,(name,model) in zip(axes,list(models.items())[1:]):
    ConfusionMatrixDisplay(confusion_matrix(y.loc[test],model.predict(Ztest)),display_labels=['ham','spam']).plot(ax=ax,colorbar=False)
    ax.set_title(name)
plt.tight_layout()


## Hata analizi

In [ ]:
pred=lr.predict(Ztest)
errors=df.loc[test,['text','label']].copy()
errors['prediction']=np.where(pred==1,'spam','ham')
errors=errors[errors.label!=errors.prediction]
print('Yanlış sınıflandırma sayısı:',len(errors))
print('Yanlış spam işareti:',((y.loc[test]==0)&(pred==1)).sum())
print('Kaçırılan spam:',((y.loc[test]==1)&(pred==0)).sum())
# İnceleme için dosyaya yaz. Etiketlenmemiş kullanıcı metnini doğru cevap kabul etme.
errors.to_csv('hata_analizi.csv',index=False)
terms=vec.get_feature_names_out()
print('Spam yönündeki 10 güçlü katsayı:',terms[np.argsort(lr.coef_[0])[-10:]].tolist())

print('İlk yanlış pozitifler:')
print(errors[errors.label=='ham'].head(2).to_string(index=False))
print('İlk yanlış negatifler:')
print(errors[errors.label=='spam'].head(2).to_string(index=False))


Normal müşteri geri bildirim mesajlarının bazıları ticari ifadeler içerdiğinden yanlış spam işareti alabiliyor. Kısa veya günlük konuşma biçimindeki spamlar kaçabiliyor. Bu yorumlar hata örneklerinin incelenmesine dayanır. Nedensellik iddiası değildir.

## Açıklanabilirlik
Katkı = TF-IDF değeri × katsayı. Katkı işareti spam/normal yönünü gösterir. Bu bir kelimenin kendi başına zararlı olduğunu kanıtlamaz. Model skoru kalibre edilmiş olasılık olarak sunulmaz.

In [ ]:
def acikla(text):
    row=vec.transform([text])
    score=lr.predict_proba(row)[0,1]
    indices=row.indices
    contributions=row.data*lr.coef_[0,indices]
    order=np.argsort(np.abs(contributions))[::-1][:8]
    return float(score),[(terms[indices[i]],float(contributions[i])) for i in order]
score,contributions=acikla('Congratulations! Claim your free prize now!')
print('Spam model skoru:',round(score,4))
print('Kelime katkıları:',contributions)


## Ders kaydına göre ek değerlendirme
25 Eylül kaydındaki proje açıklamalarına göre, bir validation bölümü varsa sonuçlarını test ile birlikte görünür kılıyoruz. Bu analiz mevcut ve sabit modellerle yapılır. Test sonucunu gördükten sonra parametreler değiştirilmez.

In [ ]:
# Eğitmen 25 Eylül 2:28:38–2:29:15 bölümünde değerlendirme ve
# eğitim/doğrulama/test farklarının yorumlanmasını vurguluyor.
split_rows=[]
for model_name, fitted_model in [('Logistic Regression', lr), ('Naive Bayes', nb)]:
    for split_name, indices in [('train', train), ('validation', val), ('test', test)]:
        target=y.loc[indices]
        predicted=fitted_model.predict(vec.transform(df.loc[indices,'text']))
        split_rows.append({'model':model_name,'split':split_name,'n':len(indices),
                           'accuracy':accuracy_score(target,predicted),
                           'precision':precision_score(target,predicted),
                           'recall':recall_score(target,predicted),
                           'f1':f1_score(target,predicted)})
split_results=pd.DataFrame(split_rows)
print(split_results.to_string(index=False))
split_results.to_csv('egitim_dogrulama_test.csv',index=False)
fig,ax=plt.subplots(figsize=(8,4))
for name,group in split_results.groupby('model',sort=False):
    ax.plot(group['split'],group['f1'],marker='o',label=name)
ax.set(title='Eğitim, doğrulama ve test F1 karşılaştırması',xlabel='Veri bölümü',ylabel='Spam F1',ylim=(0,1.02))
ax.legend();plt.tight_layout()
plt.savefig('bolum_karsilastirma.png',dpi=140)


### Yorum
LR için eğitim F1 %98,59, doğrulama F1 %92,19 ve test F1 %95,69. Naive Bayes için karşılıkları %98,42, %92,18 ve %91,80. Eğitim başarısının yüksekliği, aynı veri üzerinde değerlendirmenin daha kolay olabileceğini gösterir. Bu fark tek başına aşırı öğrenmeyi kesin kanıtlamaz. Testin doğrulamadan yüksek çıkması da bağımsız bir corpus üzerinde aynı başarıyı garanti etmez. Bölümlerin örnekleri farklıdır. Eşik veya model bu tabloya bakarak yeniden ayarlanmadı.

## Sonuç ve geliştirme
LR test F1=%95,69, precision=%96,06, recall=%95,31. 5 yanlış pozitif, 6 yanlış negatif var. Sınıf dengesizliği, aynı corpus içindeki benzer şablonlar ve eski veri gerçek hayata genellemeyi sınırlar. Sonraki çalışma: kaynak/şablon bazlı dış test, karakter n-gramları, eşik seçimi ve kalibrasyon. Türkçe ve e-posta başarısı ölçülmedi.